# V1 comparison figures — 只读可视化

**只读已经完成的模型级CSV，不训练、不推理、不加载checkpoint或医学数组。**
原600条validation来自2个开发case；新模型只有seed17。不要制造误差棒、显著性或独立测试结论。

V0恢复工具；V1核对原报告；V2在Drive新建图表目录；V3查看并打包下载。内嵌CSV只是已报告值核对基准，不会在Drive文件缺失时悄悄代替它。

## V0 — 恢复绘图工具
CPU即可；不升级已有PyTorch，也不需要PyTorch。

In [ ]:
from pathlib import Path, PurePosixPath
import base64, hashlib, io, json, zipfile, importlib.util, sys, subprocess

VIZ_PAYLOAD_B64 = """UEsDBBQAAAAIAMRDMV2QI+WmaB0AAA1RAAAJAAAAcGc5dml6LnB5pVzbcttIkn3XV1SrpwOgDUIkdbFNGY5QW7Lbu27bIWnaMaHWYECySMIGAQwASqQ1ipinjX3cmN2I/aD9k/6SPZlVhQtJXXqmo22ThUJVVlbmyUtlcXt7+1QGo3YSR0txFebzIBL5fDYLslDmIhmLYirFOEu+yVhkEg8/TYNcvngrfumKNIySwt3a+pCIIsmGU0fM5CgcolOQZcEyx/dkJCMRJcEojCeOKLIgjPFJJJkI47HMZDyUrjjHFDM8QVs6L7bCXASxCCJMN1qKYTJDoxypsdqRvMKI1AgK8yQWr89+ccVJMJyKcTiZZ1KAPpHEUgQLmbtb29vbWyB/Jnx/PC/w3PdFOEuTrMAkcVIERZjE+ZZpyiYYN5fm+zC/Mh8x7DQKB+XXYhaZz2FiPn0BRebzLCim5nMaBcU4yWbme77MFVUpOmFUQ9InfHXEJ5D5KcnDBX3dqo2Xgt9EQ/XRnefSto4mE6u13s9Nl/RJgCFpVJjn8XyWLqktTre2Pp4en5wKT9hWSvu6L/1sbDlCfXsx8tNonlffJ+XHbmfk50UWDgvL2RLmPyuW134m83AEMdIjUdP0y4i+tbbeH/148v4M893wS/VJ+8Ji0do/EadvxMU8xfAymF3q8ZsUmc4vjtuKwnqfSfX4rbjQsptPgwxCNEBrc8jaSsxr3c5xWzWJiyQaQbDiICrfWl0j3vogr4VpAvX1nnrputNP/3asnt9ufTj5zHx/iGU/n5yfvnt9xn2zWS5BbIKV+PM4LHhrZsF628KfySD2U9oe/hYs1JdqrwS1p1kyDiPwf9PAy/seNwb6dk/P1tbrj+8V+TNZTJMRhn6iF0Vr9YfgOM8Xgw/DJBvRO8cn78+P/H8/+dOdC9+w7jpND67v8ct7aIVOnd2G9FLSbevzNIlk+yqJ5jMpTn8+O8Ebjbafj6jpvZwEw2V7IfRMuusKJaBFd1yudDTt3x47wEIQ1SKV2VDGRTCRQmZZkmEJn45Oj7BFJ6esqhskfnevs7+376yIeLez193bu906e/3x0wnetE5Pjt77n346Ojt58dY/OT39eOq//nh6evL6/N3HD/6nd+8/nvsf8OfzEebiTz++f/fh2H/z7gNePD85O7e2fgEV6E3DpZMX7atuW1mpvN11O27H2nrz8eO5xrCDTkeMyEAk6QxLElqexI7oCRazQwGCxRSGJe+LXAIQus9gPf6K8drUgTnCZk+9iu+EA7n7a2xtNTgYZDHeNiiTSW3MHAGbIq6DQmaH/DEQgyiMNYaIQuaFK2Av5+ibFbB5xRJWD72vsCIY3jSNQjlysQdv3r394+kJbcDFFpbW6fpKWlgAVyWo3GnrKIL9DhciD64kkw+NgyGGcR2HC7SAGElbjcURiQNYOzGQEzwPCvFNZokrfgmiOWw/8HKNHTRPrtZIJjZNQINqBckO09kzdEI/Nwr6PVTmwUxTBuM8YnLwNQR1rjiDX0Guw2AJOfvzgYCXMApzGNalIN/F0dweJvGVzHIYdfgk4u2ypGvXBxyUou4rUa9r3aoaMKE/k4ZECfk0qx0ETTGVmkB0eHveTmXw1aif8mvKpxI7PA+AG+KrlCksMgw8+x7E+EgGeSG6PyiHCw1vz3mnBA1IAkPeCjZxJFOJv+KCnDWZhWMIi0iny5y9rgHsJdyeMC8Xvecvlk3owpKPBjm2o5DrOKF3i2cOTC9eLDaHNnySJfNUjoyrBZdGLHaWO99cofgoFt5nRyy9nxzxzTtWHIjgVElh7DkZdwgdbV8mSQHk6NAM19knK5pEV5C+fBZEEeQgk3CdwiuJzR5rHauWt++bx/5wGsQTmftXuW/MOpb6WrVWo0AmVky8Wni303liAxt2yqfdllP5nrmEWwilBttpzZLcTQUN2BysnMdWyhIl1yCbmXaoN7l8Ng0nU/OQmQPEITUryatkDAwHHtYa2mkCoKizgXWEOpHUzwbESpFDTWXJnwN/EiUDgLaRADjgI5mM4WVYb/nJjpEBftLmR8SPY9qbCAEB4BIyOSbiMDRggGSW/NdCzgCnpADl1hr/DXsrF8NoPiJJIS+3mJK21oekaIPIzyQ5/rnCJxKCXHS67c6eK35Miin78MwhJYR9zd1IjguaZSALQKcrjky3b0mCEIRHjoKBBDgx2gIUYqIc8DGcZ1dA6TycxNCeYQBGimEUhDOHgpJPGAQiMpljt4HKFSef+dXqfZIF7L7M0DiPIUHs21Ud2rEsrpPsK2b5JhU/P9KCiVl4QDaIvmUs/XW26vdyV7ymgXPaV2lYCEbnMmsbG8GqMod0iCIYRNrwqK7hDCsFYiqXt9yeawnxK5TYZBg+nMkdLKUIhxFtbl5oVuutg1pKYna5WmxhAj7TPJAaCMQYoCXrkRxx63Jra2skx+Rw21lw3Qdgw+q1RPuVAB19tqJg8jyLTUjlomtv/4B6t9ypXIxCiH9ht/RAX66zsJA2RUp9HSARlMq+SAZfwDge+gPQSY19HUJuqLObACttawHXEsQlFIF61rwYt59bLQp/xv3SolPY5o4QF9k8siPG9E5O0WKQD8PQewPrLB0G4Ljweg6iU0iiHwexetQqhxq7ilwLHoNZAQWyfhWy1pbCtBfzNJIXCJyLC3DIEfTp0lF8u1REIoqFVQ5HgCCWBNjOtrKajoDT09bmOZhMMkIjeUi2cRigaQ7JlyMWM5jgOHcpIKYhiQg4F0QFE6SWEI4V78Lcz5czGIKvdosUgzddPyC8sFsV9yDHwDH2Gk5ITW3rdRWez+YwbANJ5kvCMrEOxlARPTqH71Zz7hxBud3CPz7pj3glen6n06E/9815skghDKT8ynZUzKA5lHoATQ2rwMrATAzBAyd4bt4q5rytn6EBMu5RKsA9xs6ccoMdJu4Z8D+evPtIguuOMOwI284C1ga8WK1WuSyaWw3kwmJHoxgKlRNbI0jo2oOW+M7jJ7ksNjy9l/M/h3muEywjiBUATq2/3H695Jnu5wmahCI0iKLYPKFZhX7nMdNn8q/zkMCn2gRNAAJh8VRQuOd+gUGz8yTDptl6bMMz47gj+rjlBrK6WXJNdkIRWFFBRKIjnl6YCPOyfEjMp+e0A3hXpTtInqmRB+OJ+o0gaX1df4y/xsl13GSrmkwtiMarQAAAMOPISdPT5+e3FUiwF0iev9CRcJOAIlv2FchhlHGUQB9oeXjlstXoCKyWaSHsilZHnC9T9bHV37CSD2Se4IvDddEejFoAxm4p80FQ2phEC/BMqT8CGcI3po6hQdH5UnSaa7hDQHjKEhXUYGyRYrLRypOqKGrSAa4yD8AUnnSVnwBtAwLg7IVdSy704M3ZtQwDw2brsknyYzj+KGZAfw0lj2DK2LrBPLclU27Mu7eA8YSnywG57PTPEDQNkymUplTlTeyBp6iJqpRVrfyCJJG7oLvRbKX3/JzBh76zqtyLNSaI0+Gb9mSV4lPwa+I3kcxC+GkjbbHgNkTRIBh+LeFXeQPEhH5JJi+kpidMzq1DWF1a1abrTxsGeFmxo2xhIxgeNrB6wykLyEEOw0Z+sZYJVACSzAsKvy9LBDKo8eHkc3+TMldZq+a2D+CyYCQzLS/OUVPzdnDDqpgNxEvvcUp1asKHYRnu8C6QKeT1QHDKSMQQsSo+WKwbpBRe2iu4Baw2L1Eecy3/KywFJHhISrhGsPmvHMVn0UT3gXoXU5VNV2gyoac/klERUGN7cM+wK4JgonxOSCGqu9oZUDB39/vN6Tipx+uskgUceuUW7QpW6IJHObmZtqVSfkLCBRSWTgpqdEXUpdKDtw0pB5u1/LKf6MOtaLi2SogrgYUvSpYYbewmir/xs3/C53U47AD3PWvN/2VS6k7OZ26wx3p29gM8RUhr5S3l706VV9RqaIUx2LSkBvvrb+KpffO1Ty/MDOpeknvQfQYfilgewjOAT4gAo3zKCK35rtqUGqo4ksi8Nc638ruln8dBmk+TYjNMICLUz8uNKJKoryYCY7qy/YyZTr1Ln1z5uVIcZ6RZpa+Tqw0hR12nqAB+KZAt5LyUmUmHbIyNynBU3nktoegTSq1EEGYIxe8x4moOnUuwonx/mVQAiridNQxjQN2IYhtdkgABiYHLCrOcitC7kAxk4D3omB20B60dUGbTlwEMMpi6Cyvc6L5COXWvN9GE0X0WeRglubSJWu7qYxc92kqHSOBvYEZrHVUNDw0GMpvgqPYbrgjmMh3vs4xnnNYkt1ula/Iqhs9kyh4vnZ+aXWRjSfhgzkaVuS8N5arPbEhoNe2nVQkI4Vcpjm2VHyWZZI9XE2Cyp6Vg0gTEKrzMHGvwyKrvgq8A36+SUXinsU3Nd7E/lIJKxj7+VxbeTzKfT3T9kGLqsFhiiPNsDi9Z6e2Vjnd9zhH7imxb/YOJM6OnTb1lHaUPpY5SvAZvLGArWEwzKXV8qIYSFM3mpaNF+KlPr9lzKShfl82prQjHGKSuooYUE0ZXLWpjVJhHBxPVzriAWGL06cmnj6fn/unJ0fGfXMo/UOObd+9PztS31prC8miV1KUMC2bCHXraiHoo3K2idVpMuhbVp4+KtTfLuA71sAE6kid2XSHAjkK4e4a7EJSAQu3VKGkWxOFY5gStnHyhEoHcrnFwp8EMFZcXclHYa7mcdU49jt81EAhVuspe4WerkQ5gz9jQ7U7gInOfh/iksECzg4bitBcF00Cs4XSVMbQh8/wetmxYyMPcwQJtNTITbuVDCLrFK1KHdQQ6ted0rKcds4qV/DGSwA6yyyoRSBpbMmB1kOo8lyaiw7m1HlrLKauqQjXu2rs37vicJaaExFCkSlMAqZrPvL4yvoDN2mxG63xdERn16kbnocKcBv7eM5iGNF+luG2gXYQgl72PfD6ovtIhJ4R1yonayvk4cJ9phmAAylRFhauHwj+kuJ7d7brPzZt604MFxbLhxA1GI58y6fZFx+2hF/7eo78P9unv/Rc6xKWuLESwkHv86AX+ZurgcSXwgGmm7oGjM8meNUgiOs6/CjyrSFLrrmGev+hVC60P1Xn4Xf5HHfHW3nzuHqhXB0lRJDPLLNjNU3i4COhoyEsXcax/Bds2AATW8rS1fhkt5O6eemtBFvyHhdlGskg2t8GbL+0Q4mneRL1V34tPH94yLOUQcZnvnMM4Rokjzn5RzXIUcuZeXCHYJzCOgqGkY2s+svh0/GaHl0uZBDbYbolz+Xw8RtzNSJfGE8ux8qtJHc+IjUQk/rVB4s7YuiHybt0b9eotNm2Uhl6vgx0YDJKFD1WawsW3CubH5mDJYLl3Y50l4+IaigGbrc/nbxlEeXTPY6qUd35jvYbWYX1VV6DxMQw7GiiC0eERybTy3UC08d39QZBpleP0To6oSpYiueAjHt4ExX7NAT5KhyvMvupMuaRP+FXm3qz0fi+NzDlKVxr66Yi1BMff9PFTefKEFspxtNUhLx+ma1Hkc3WPJA2fpnacunSiNJH2QYuPL3Kjq5570KteudhVojgl02BbOzs1PyDkDd9z9lt91Tds9HXdmhageVmEw6+5Xc16oUpSwJFVNjRUshwkpIN0jEMHynZj6AVMvA3JIeec1tJ60gWirHShvbHNFlXjdx11MpcGo/pkCz62nmThyCaTQgcs6TTw3F6vMSz1wssIGalXxRvwg9lKa/oWpjbxR/G5phUYg7GF+j2tEe92us8cGoKMkr+0W0/NZ7VFdmsH+AUdQve+uz++1ZBH2QGZWRu4VyEEySaLppboWfBVajHLtWTrpE0NTDTNlJGZfYU90dyvtGFTPVRXHjhWpwuR3FwZYsypKmmo1XOIiz90O3++aR/c/kGoIQUPeWkx9auVJ2u0rNdhKVJ6q6SY6o+NlODhg4Q0SkvW6KjXvHXRexfzP1DpVC9d/HlTD3HxwyWpvS5uII2m2EB8NiRtrCpp3YErVmcPg20svyDBVcGJmgC7rs6E67VoGwBpPI8iVa/Bo6zj0SGVOdUKLDgvs5IvLF0O2NV9RXwyHkPfGEbbbm/X6Tj4W+El4JTb/5Xyunvr6coUBimqSgGUUH75BMK1gl9V6odIq0Dhi2OT1VBLadFzybk52B6bQIJ666eNUzVCBa9pPdYnrZIcxGVvE9A/VWM7DPcG7Xtdh/HPG1taqm62F8tv2xdfLm9rGVkDalcNRFsBtAaoPVUMIzTrOg+AGWFZ390lJKtcKqeGafdaknsMSR0JDx9jQzTRXbf7jF9YtQOVGTi8xwQ0jU4TWBpadjfCbDJK2CBOBQ2TyNt1omSIqCpNdf2J5bDjVCR+AN8pyTyspu12d1vOmOolkkZlQNMgbKzLqkAjWHgVYuxDp1WpjphwpQxVtyFANMvK5ASmJF9FitO7C54wYBeh2P/9r6CCJ7EjqMT5t//8h6CKp7sLnSwjws/c5w2um0I2+8Ld3QNM7DsIKfDnQIcVgyVUyLuxs+p41lGfadTLVh9f7krhKwmjSM9g1W1NvU28X6n1h5PPa4rM83N20lk9VKrOax6jzvaXNuDxibvbWVHp51qltV5wIvRfUmXb3d0nf/rqlddRbnQbLa2H1Hqj1866/tTtjW9/wBZCxVl4m6PrMKiOAJUqv1BLgb56BMVdxGSOyeLet3Wttb1jP7G3DgOUMnLwh3UcgED2DNpE/1yHo2Lqde/Fo3rB9e9CIEaL1dMzNvn032//8V/1Yj6oTb18D4//sRk1/mkIq8NN75Fw8/xRcPNAneQd0HOAJb8h/0L7bzvaD4I71awW1JhTqxns310xuKlaMJ9ScUWjBvC3v/93Z8+6C2i6u+SPOO7zjlPlL8qKWgSozdsiNlS23YMCVTdE7N19Z7djWuqXQOxuz2nv9tZPDNfr4O12d9+hPWjUwaOVmnutW+3IZF+B2d6FlVCkjj/H+PNn/LmyqqNl1auENMIJ/aLDtvWi27+s4cXCWXq1s5gNAQE7LU69T81BruCOeDsMKJi1LxaXzsXy0sm9F6ThNLmnyboP30hn1U0qadd7ODaIbDmLJYGaV26OfkiNw4Syg56l/CShT1nvgLFa0ggucnINA5jSTvOXvFhG0upbbXC1hA2r7z6/XUebXUpr7bkHnVIfl9R84PY6znOYr4ddiuvVAOtOv0L89vf/WU0abEYNRYdCpPujFgKoX+N7Bq0Qdh2ENgao9xUK34ENz4AN51TAyhksPjvQdwXpXlNN7+nsqQ4SVMpeQFiitkKEWnnpoSk9zc2FQ329wqW6XNVJcMWtyT4D02Q8XLqlb7LvPmuVOSDvorrHchErwx/r8o3LMuVS2nplT6D3DeO+t3+37elVvvDq8DU71H20J1wlU/ZL2TRWqmJ2jbkV76x/0fw8yklpxBpl8qT3qGjDub3Dveh2N4vkvRXXJp3CdyCWPt5I54VdJVCaJ/Vo9/iADh/08dPYo7yopS7Z+LWjJnNcwkfK41q5qzqHeIMvEOE3oGJUVtSJ8mDEBNqgW4YpqLGewqzYs3HLHHulKZ3O1A53MMmDxzfBsIDJ8W5St7TjRcKrcYOc7eLCVu5WSvtHiaOMFNq2nqgSitpJ4K1ZoRrUlHtpylp/u9nAlNv7jmPeqCWrPeDbt7OqCHUeZ5IcAklnsAAVddzq1tKaWJFTL9vThLhUn5bXy4xTr3Ev1m4UApg1mloedb5JSVEaM3Wpzp1PdK1ff7VU4Wf0cNFnHozLvVT1NbUQvaB7LQXLEQarU6KerJ22BvHSXjRb+QIN537UK3TYRDl/DLL4jobWLQ+fMupDWLX7tBGr5OozTj3R2sFmWa945wZzb+0kU9Xr03ID9ClJ/cSypl1nv+OsUmn1YB5GI7t+8JhfmWOW+okLX3/wv4bxSB28NM/+acFVD1MBbFt0i8I3VRB+WSVBvlmt0sDnQ8D75P5YDiMq/tG3MPhKRMz3ScDLKByGRbRsHEFmwbW3egLZXKQ+dFQuurexwLG1jmhuJfRq4bqgAs4RX6inkX11x8Lq62sZ5PhGCR4lI9l4xqP6jBW+v3IAvnYrtsZfvF19gWur7uBZfXM8ZLaEZJovAjR0m6/6e2to3pBe7nNhleUilyS25lsNnk94dLNJukKl0Nu0Qwvu000aEbCrorVF5VuTbFlXmVrJnma6zsdrNfXYwPJq/ORrPQCravtYEWq7DHQjQSEhUAX5XJ9ANX28va6qAyNmc1H+ptHui+XUcOZIgYfVXy46l9qa6As1PJbetEp4dS0KGU/bMLfadyVP5MEr8VsRX5ezLGXvOPEhar4WwiRdGoYpBVMM8JhIXmz1JhUJ0GURqqLks3zIsqkQ6PecWmlB/6DTUeGX+XEJn+71Wv3us5UiJKPpLJaWbN75re646l+Y0Bd/V6IRq3ZvF4Nwaba+j8VFXMpB58M/LuOqXTIrb0mtD0nhWl+Xj4qyfFSFEOoWMU2qLnu+Xa6+X1bgYQztN0fN68mH1aXk1Zer+gu8Xbs6TT/dgemTbK/DVT1RkObkVIPbq0PUnNA+g2u7LDGrXYnjG3/svGFJEBsMX6UCzJUz67YaW99Qo1h+WUyJPvMjGq5q8DXG2BR+029aWP04dX3T7Pt0NGV+CsPq134Wo9anPiHdCkcwGsvm1SHe2OFUDr+qANUpf92E7g2ZXzwJZ+rSJoXy/KMnBDRW6fatnvoZDWUl1E808qhSWs+2vhdv1Hb+0uU4Uv86S6Vzv8a1a+mWuk5qMiiUUuF4rLx1RZfxjUS0laDWQwn6LRdeKiWFN/9QS322J0+onqcSn/LKDlc69hrX8PXVe+IsBm8r/pCWcqVD7VazuiXfplvyjpLX9ijJ6RK1GMK9YUbzZU33yZPG4sfWu5idnhLH+uIvNzWLdPsXV6guVJV59tMRDB530eYQz+urW4k3a/dTBUJfsvp8C69Klt8p9+YqqBrHbW7Zj0FG1p5cU7JItfvuuSu029Xhq+58fc54Feaaq07ccMatQf6HBhIRuRR+xoW5j1vJN99LPlQQWD8T1gF2/YJ/k/Sf6fd6kkFOv1xAC+2z3KgsAvffoRNdmpozFQ6cuqK6U86XxlcOQMvr0QV0Y/XQwm1Keli7iwphQVwc5lNiEmEVFXdeh3FMszYgGAy5xgTVyau6VSyS4XCeqfVpfR15SgvLKIVSFY6qCxkGKQs81q8d3MqPmY2eemPr++/FDfe9pTG/u9BfLm2tnDu6GidFvEU9bvSQtxWLlXUmx/nnE3c2gt+snAB2nGcjZ813VhlQEiXvwnr53SgZFstU8s8UvXpJf4sIiONty3j71Uuq5GHBQ+TnbfMIaGUqXwFs1oDm5Y56VkK/9ZJ/r4nTbt72LFi0OePW73Y7nXRxOEOQEcb9XiYha/MiOaRovz0OZmG07B9lYRA5OeSunZPDd0gJu7aueeu6+/vb9Xmm3VclCvaNfO2UO1inEV1rL6avCKA0KjmbEMnZiEj3/LQH3bVnVHKFEUZStdoPfDB0bYArcVRqHHtDOefFSritIPblTtpcxDquWU9pO10J7U+lXcO41lOrlilTer9igeuEm58tWKkGWP8ZA3VPoeaSNbRKkXz5ezVFBeRPqY7g5bT3qrkofh3LwZ72XjX9jbH1MpxNjORpqet0ftgWQQRZvlkf5nZb5NnQ215Xvu2VsYnfTUL0CpiUcpVMurkngAekCq9AKqma1TpUqjvhn69YutTa1F66Ga7q+Hmg1h26XPfSNwTSzo2lKmnhdJVwbnwMfyJjPhCl38Co6uvX/Xk40sb56NONYz0RPCp1HQfNF7USw8vam6p4v3S6U5nRK+R5q8vyukMp2us9alTAfc+HIVEHqfLrTqkebBIW0/mALg0UXEbpE3cKaTrc3sWzehbLuT9/xuFvupIZKXNq+p5yM7W2llvTyefNcezaDTASHBMF88/OZZ75ATr3KJvMCao+cbs9gkBmIQuj5/tAdwTntfdUia9+xbbabQ4DyQJwnOaYa5lelfG940VQ98+9qICozbkAuLdJOJRkin5HtuVy02yBp+fjf2hGkz5PM7rlanSjbzkqbRRQCOwELmWRA7cOj62tLWyXz+Gr73ue5fu0A75PwQHtxNb/A1BLAwQUAAAACADEQzFdP3MKqwACAACXBQAAHAAAAGRhdGEvY29tcGFyaXNvbl9yZXBvcnRlZC5jc3a9lNGumzAMhu8n7U1YZCd2bD8NQpDTMrXQAd3O9vQzND2TNuncnV3BnwTXn/8/vZbtPA/Ncl1Lu27zUob2Po3b2ly7vxZe22vppvbWb/tr91rfbsv8Ml5K+2+Fn+/s/Xpnb2r7bi37cyn9vAxr051OSzl12zhPTfk+DmXqy+dPU/nRnr8O7fLSUECRpEmByxfITQzGyMlQ8dASYtaMKkZNTMGQLAJqg4H8iGTS6OfYNQBQjJz0TSfCnKuOTQZoyrd7dzmafIxkfqmdutyWsV+b+1oWX7vNy+ZYEWIGw9wu96k9j6ezw5d+XB2n7edpnS+VZinrOOy1H0jEIqiAFUmFxSALVCQ1BoiUsYkchFU5+rojpcRqke1o2ZwWogCoHt+5lqyKBo86H4h0O3s9hMHt9RKbI6Uo4qNHOn46BYSczRcfOocsPmwldaQYkqBoOlxKyuK2KFck9Q11KHkiUTLNwPpfkLjsBknIhLCbwpXGxDXHahBi8DgCS5ImQeDkxxN5b4GZIWKsmZOQPGDuH9NTx+jpNEgfnbmDxob2drmvhztAyGz45g6QpkRPd8yUsgGm3R31ZGokNyeKG8Nkj3Y1JHLTWGq+NBD5nNAI/4s5djpImDNBzcdBYpaTxHp1fPxMfqn1D0lK+9XxDfBkPXNm4DkTEnvmzEOm6H8zH43yG1BLAwQUAAAACADEQzFdeeW4gWYHAAC8FgAAFQAAAHRlc3RzL3Rlc3RfdmlzdWFscy5wecVYbW/jNhL+7l8h5IukW1Wx96Xb2KcPReItFijqhZNte/AZBC2NbW4kUkdSjt2i//1m9GJLtvPSxS0uHxSJHg6fmXk4M+TFxcUdGGucwoDDVysNK27BWYqtLTQYR8l0N3IkbECTiHa41nxnLuM1xPe5EtKa8OLioieyXGnrxGbTvFb/UrEICyvS/ahq3r4YJXtLrTIn53aNcvUM5xN+NkJm3Z5sIcuXIoXmu5DCWkRfqWm+wkzF940y1B2ve73pZHIXkWKPMdLAmB+ieSrdgOeHOdeAhswG857JIY660EMaY7RENTVVMbdCSc/NV1cb8YcbkPbL+ivMd67fw5djLZlKCpxd6iGNHj38ET3DVPEEdAhbXKiS81CD37udfJ5ej6NKf8Itv4xVhmgF+o5pIP2QhOh0t9eLU26M86swBU/LkHp7h9DnNTfgD3sO/iWwdAzYz7lnIF3Wg/RHn6HN8qhxNM4kI7je3QgNsVV65yFkEuNJcp0Cl0WlhaaFcTXgdxVqpWzl+72g5BnUehB8tBe7dA/2VWZ1NOFI+KCFBbbYoWFe5R4MJE/qEd/fW5gVFplcrhksCxm37NTqwUSpMNYjjTcitlOgAHhChbdWC7n6OPH2K5bqLWwtavf9rhJS7NHbYfxB2PUBrsoBifLgBhIeUiEhcl3f4cZZHtCUk6IGyW9kn/YQs4A0IUeZiBaY9efhPezIxNFD5YV1BXr/TWItMGWY+QaSmikn0Ub/FCb60zUxonSHRN7b68mnceAiWJaBXauEtQhHrykg49zhnS4AxVBzrHRi3OH3/T59b3gqknJ3sBgJhz+8/mu/oNeK83T8aTK9Y9Pxjzf/CikVuH4d2tLTNBImRZYbr4LpH5MK1tysvYOl9cCRje0lP3z8eXz79Fp/ytILZs291kzpdynmLJV2pCOk4x3zNThn2F8tVtJ+ZLDlsWVGbBmXCeN5rhWFKVZpkUlzbANFNGARASthHJbcc/TIOZgIQNvxfzATeLbAmFWsCEjFZHoznj4hj4mhlJ6VJFh/QfYs3bkf0DjNv578fHtsTyaMwU3TUEbDF8wVkJxNL/W2THm2SLijzVCb2fC7wfzcBqpwTblAJnm/8rSAsdZK+8PnfbHHlqD9AhM2fC26VzNNm++bAZRKLoUkNj6GjESXnjb+sEQyc3VmgBnMxkgaSvLGnUeu5NIdacCyLRH4WbOW38qGB60w/LFaY5p5uRmH9IHw311d/d/gYxbQnOVabIgn9TbEwJAtBOgpWzppnBKDpsRAzJm5jUZqcMjGmwn7ZXLHxr9TinA7M5+zfPS1WeAXZT9Sq9KGEhirq5TQFPO2aBdlS7brNA0pJvoNsARSy8tMlucswaqKqaAQZn3qtZfYMIrXXK6w6FVS9Rr14FGpjSNJCVwf/F7LOQILwsytdjx6PmonMweh1r9qEZe/bjE5cMny2LpnnfhjmilTJ8h45vIFdo6FrU1350H47urt23f9N4M3QZ7yGNH/8ESKPdFQbmJU4+agY+xE+QoDVfbWL4Jz5CVWa0GFP4RX/e/fvxm8ff/6fYPs3VmVVNC9/5E3T9OT/wTIf/aPiGUkz82aNp8qZEJ1hccx5JhRqNBk1NA/mmNexLDT8nY2pf4jGrwawHeD/l47qa00wh5lScmg6kS7zeHLlgj7g7+b1Z6HcPBlq/9jpevgpL8oF+w0ip2sUNGM1q2bO2AnXWXZJzUIgkeybAeL5Rky4OlifA7T4QhQ9m7nmvRX7r+l+7dLxQuN65okZNMTs7KJ3tvD8MAsGS2NFNR0dGLn2tNzZubt49BpNzkyUdmw0pHReHnndDKqOfdU6z6PPvDUwCh/rN1uakKnw/7WjlSFzQvLUrGEeIfHyLP1lt/j8Vms6F6ionydnwKc7Q/xEWb3iaADkYcfl259icHoDiO0W9tt+12jlvYB91GJ4Jgw5b1BqBYUTiJ/4Gat5bEqigQYLBGujdrAjhoCxFEmo0Uh0mTP1qAVYZTAUwNdrewP9Gx/D9NCdbaS4OTgjHoaflxlV+cjUf2Ap//xFou5aYX23DKdXU1HoGcglxQBLZa7OuhkBNaH6phHRXAPtfEpW4EEzYm9XeVVmI+bBLYxLEfuwqBf3Y10w15JHat6KblPkHd5zNN0j5qORCkwvsV36o7+wAM6U1qshGQLrr/qnIdbE2Q0m3c2hpB0iWQ9XLiMibGQHfHwJAwpSJIPCZ0fDLrOwHpPOkKMibaGXON5bn+APO2/pscberylx3uk0/BEeaOY7ixWYNk2FZnn41fQP2YHyBDPvyCRVoR6VN6gpTaMU2WAFD27L8s8092QtUOqkGUnaePxzrK9MXHKI+1Xc1AGGcw0WnVomEjzh48/fZ6Ob+cnxzxGFKf2OOYpVg4LGlOlQUYwytmwxXooMrqHPCYG8TbyTi4Z26n/ic6/uTtVOsbOn8TPNf0G+EJpWQs8dZDA4JhikacKG5hH1ZVSdpfC8wqBdhfuB68R7SH9GKObL8awm2TYt+COYe7wcMGLA2jzfwFQSwMEFAAAAAgAxEMxXfHBLrLyAAAAVwEAABUAAABwYXlsb2FkX21hbmlmZXN0Lmpzb241j8FqwzAQRO/5CuNz40grrbTqzwSttCqGNDaWY2hD/r2ySS8LCzPzZp6nrus3Weo43fvPrp+/wnnT522sj3irZz2oQfUfu6iMN6lN8mxPdwi38XeYf3aXp2BQsyiXikNTVGGMBbx1nqxIsBzEJW1dAUu5FK0tcoxoHItRBAehhea4xkuavue4jHW6XxeZp2WVPKS6HRxw4l0x7DgBIVttxHPAQpxjStoZldgGCCVG8ABsIGcwjaqwlVH/nFXqWi/7vb6XvodkTU5aNx1sdntDHXXxGdFJIM0KLfmoLCrgTCZlwojsScBnUZSgb/mv0+v0B1BLAQIUAxQAAAAIAMRDMV2QI+WmaB0AAA1RAAAJAAAAAAAAAAAAAACAAQAAAABwZzl2aXoucHlQSwECFAMUAAAACADEQzFdP3MKqwACAACXBQAAHAAAAAAAAAAAAAAAgAGPHQAAZGF0YS9jb21wYXJpc29uX3JlcG9ydGVkLmNzdlBLAQIUAxQAAAAIAMRDMV155biBZgcAALwWAAAVAAAAAAAAAAAAAACAAckfAAB0ZXN0cy90ZXN0X3Zpc3VhbHMucHlQSwECFAMUAAAACADEQzFd8cEusvIAAABXAQAAFQAAAAAAAAAAAAAAgAFiJwAAcGF5bG9hZF9tYW5pZmVzdC5qc29uUEsFBgAAAAAEAAQABwEAAIcoAAAAAA=="""
VIZ_PAYLOAD_SHA256 = "d3c9bdc7a508f003d2c0252811984e184e38cfa62bf9f81d845153a6556fabd0"
raw = base64.b64decode(VIZ_PAYLOAD_B64)
if hashlib.sha256(raw).hexdigest() != VIZ_PAYLOAD_SHA256:
    raise RuntimeError("Visualization payload changed.")
TOOLS = Path("/content") / ("pg9_viz_" + VIZ_PAYLOAD_SHA256[:12])
TOOLS.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    manifest = json.loads(archive.read("payload_manifest.json"))
    if set(archive.namelist()) != set(manifest["files"]) | {"payload_manifest.json"}:
        raise ValueError("Unexpected payload files.")
    for name, expected in manifest["files"].items():
        relative = PurePosixPath(name)
        if relative.is_absolute() or ".." in relative.parts or "\\" in name:
            raise ValueError("Unsafe payload path.")
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != expected:
            raise RuntimeError("Payload member mismatch: " + name)
        target = TOOLS.joinpath(*relative.parts)
        if target.is_symlink() or any(p.is_symlink() for p in target.parents):
            raise RuntimeError("Symlink tool path.")
        if target.exists() and target.read_bytes() != data:
            raise RuntimeError("Existing local tool differs; do not overwrite.")
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
for module in ("numpy", "matplotlib"):
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", module], check=True)
# Run only the lightweight visualization tests, not the old model tests.
result = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(TOOLS/"tests"), "-v"],
                        text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode:
    raise RuntimeError("Visualization tests failed.")
spec = importlib.util.spec_from_file_location("pg9viz", TOOLS/"pg9viz.py")
pg9viz = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pg9viz)
print("TOOLS:", TOOLS)
print("Software checked. No new model results were generated.")


## V1 — 挂载并核对已完成的V1报告

默认指向本次已报告的目录。缺文件或hash不一致就停止，不重新训练、不手改旧receipt。检查范围是comparison.csv与REPORT_READY，不会打开报告中的医学图像。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
REPORT_DIR = Path("/content/drive/MyDrive/ct2dose_thesis/phase9g_signed_real_v1/reports.local/20260916T212513229830Z")
OUT_ROOT = Path("/content/drive/MyDrive/ct2dose_thesis/phase9g_signed_real_v1/visualizations.local")
COMPARISON_CSV = pg9viz.validate_saved_report(REPORT_DIR, TOOLS/"data/comparison_reported.csv")
print("PASS: completed real V1 report scope and selected report-file hashes verified.")
print("PASS: aggregate values agree with the reported-precision snapshot.")
print("INPUT:", COMPARISON_CSV)
print("No record identity, significance, or blind-test claim is added.")

## V2 — 生成7张图，每张PNG＋SVG

这里只读汇总CSV。原run、checkpoint、report不修改。相对变化图以Phase10D为固定基准；负数=误差降低，正数=误差升高。不是百分点也不是综合分数。

In [ ]:
from datetime import datetime, timezone
if "FIGURE_OUT" not in globals():
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    FIGURE_OUT = OUT_ROOT / ("V1_" + stamp)
pg9viz.build(COMPARISON_CSV, FIGURE_OUT, "saved_report_csv")
ready = pg9viz.verify_output(FIGURE_OUT)
print(json.dumps(ready, indent=2, ensure_ascii=False))
print("FIGURE_OUT:", FIGURE_OUT)
print("GALLERY:", FIGURE_OUT/"gallery.html")
print("Original experiments remain unchanged.")

## V3 — 查看核心图并下载

ZIP保存到图表目录的旁边，不写进已校验图表目录。图片图注只包含模型级汇总，不包含case ID/医学切片；是否允许公开研究结果仍需自行确认。

交接包已含同一已报告快照生成的图。Drive生成图记录了实际CSV hash，可作独立核对；不要覆盖源码包的snapshot图及旧manifest。

In [ ]:
from IPython.display import Image, display
from google.colab import files as colab_files
pg9viz.verify_output(FIGURE_OUT)
for name in ("01_volume_rmse", "03_x_percentage_error", "05_relative_changes_vs_phase10d", "06_global_profile_tradeoff"):
    display(Image(filename=str(FIGURE_OUT/"figures"/(name+".png"))))
DOWNLOAD_ZIP = FIGURE_OUT.with_suffix(".zip")
members = [path for path in sorted(FIGURE_OUT.rglob("*")) if path.is_file()]
if not DOWNLOAD_ZIP.exists():
    with zipfile.ZipFile(DOWNLOAD_ZIP, "x", zipfile.ZIP_DEFLATED) as archive:
        for path in members:
            archive.write(path, arcname=str(path.relative_to(FIGURE_OUT)))
with zipfile.ZipFile(DOWNLOAD_ZIP) as archive:
    expected_names = {str(path.relative_to(FIGURE_OUT)) for path in members}
    if set(archive.namelist()) != expected_names:
        raise RuntimeError("Existing ZIP members differ; do not overwrite.")
    for path in members:
        if archive.read(str(path.relative_to(FIGURE_OUT))) != path.read_bytes():
            raise RuntimeError("Existing ZIP bytes differ; do not overwrite.")
print("SAVED ZIP:", DOWNLOAD_ZIP)
colab_files.download(str(DOWNLOAD_ZIP))